### Step-0: Testing Connection

In [0]:
%pip install confluent-kafka

In [0]:
%pip install confluent-kafka[avro]
dbutils.library.restartPython()

In [0]:
import importlib, schema_registry_config
importlib.reload(schema_registry_config)
from schema_registry_config import *
reg_conf = get_kafka_registry_config(dbutils)

In [0]:
reg_conf

In [0]:
from confluent_kafka.schema_registry import SchemaRegistryClient

sr_client = SchemaRegistryClient(reg_conf)

# this call hits the registry — proves connectivity + auth
subjects = sr_client.get_subjects()
print("Connected to Schema Registry.")
print("Existing subjects:", subjects)

### Create a Topic

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import *

TOPIC = kafka_conn.TOPIC_REGISTRY
conf = kafka_conn.get_kafka_conf(dbutils)

In [0]:
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient({
    "bootstrap.servers": conf["bootstrap.servers"],
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": conf["sasl.username"],
    "sasl.password": conf["sasl.password"],
})

futures = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=3)])
for name, f in futures.items():
    try:
        f.result(); print(f"Created topic: {name}")
    except Exception as e:
        print(f"{name}: {e}")   # 'already exists' is fine on re-run
        

### Schema Register

In [0]:
from confluent_kafka import Producer
from confluent_kafka.serialization import StringSerializer, SerializationContext, MessageField
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.schema_registry import SchemaRegistryClient
# --- the Avro schema, v1 ---
schema_v1 = """
{
  "type": "record",
  "name": "PostEngagement",
  "namespace": "com.poc",
  "fields": [
    {"name": "post_id", "type": "string"},
    {"name": "type",    "type": "string"},
    {"name": "user_id", "type": "string"},
    {"name": "ts",      "type": "long"}
  ]
}
"""


sr_client = SchemaRegistryClient(reg_conf)
avro_serializer   = AvroSerializer(sr_client, schema_v1, to_dict=lambda obj, ctx: obj)
string_serializer = StringSerializer("utf_8")


In [0]:

producer = Producer({
    "bootstrap.servers": conf["bootstrap.servers"],
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms":   "PLAIN",
    "sasl.username":     conf["sasl.username"],
    "sasl.password":     conf["sasl.password"],
})

events_v1 = [
    {"post_id": "post-1", "type": "view", "user_id": "u10", "ts": 1790871216},
    {"post_id": "post-1", "type": "like", "user_id": "u11", "ts": 1790871217},
    {"post_id": "post-2", "type": "view", "user_id": "u12", "ts": 1790871218},
]

def delivery_report(err, msg):
    if err: print("  FAILED:", err)
    else:   print(f"  delivered key={msg.key().decode()} -> p{msg.partition()} offset {msg.offset()}")

print("Producing v1 events...")
for e in events_v1:
    producer.produce(
        topic=TOPIC,
        key=string_serializer(e["post_id"], SerializationContext(TOPIC, MessageField.KEY)),
        value=avro_serializer(e, SerializationContext(TOPIC, MessageField.VALUE)),  # Avro-encode + register
        on_delivery=delivery_report,
    )
    producer.poll(0)
producer.flush()
print("Done.")

### Consumer

In [0]:
from confluent_kafka import Consumer
from confluent_kafka.serialization import StringDeserializer, SerializationContext, MessageField
from confluent_kafka.schema_registry.avro import AvroDeserializer

TOPIC = "post-engagement-avro"

# the reader's schema — v1 (same 4 fields). This is what THIS consumer expects.
reader_schema_v1 = """
{
  "type": "record",
  "name": "PostEngagement",
  "namespace": "com.poc",
  "fields": [
    {"name": "post_id", "type": "string"},
    {"name": "type",    "type": "string"},
    {"name": "user_id", "type": "string"},
    {"name": "ts",      "type": "long"}
  ]
}
"""

avro_deserializer   = AvroDeserializer(sr_client, reader_schema_v1, from_dict=lambda d, ctx: d)
string_deserializer = StringDeserializer("utf_8")

consumer = Consumer({
    "bootstrap.servers": conf["bootstrap.servers"],
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms":   "PLAIN",
    "sasl.username":     conf["sasl.username"],
    "sasl.password":     conf["sasl.password"],
    "group.id":          "avro-reader-v1",
    "auto.offset.reset": "earliest",
})
consumer.subscribe([TOPIC])

print("Reading with reader schema = v1 ...\n")
empty = 0
try:
    while empty < 10:                      # stop after ~5s of no data
        msg = consumer.poll(0.5)
        if msg is None:
            empty += 1
            continue
        if msg.error():
            print("error:", msg.error()); continue
        empty = 0

        key = string_deserializer(msg.key(), SerializationContext(TOPIC, MessageField.KEY))
        val = avro_deserializer(msg.value(), SerializationContext(TOPIC, MessageField.VALUE))
        print(f"  p{msg.partition()} off {msg.offset()}  key={key}  value={val}")
finally:
    consumer.close()
print("\nDone.")